In [1]:
import pandas as pd
import numpy as np

from src.testing_validation.model_test import calculate_rmse
from src.helpers import fit_cv_timeseries_model

from lightgbm import LGBMRegressor

All Data

In [2]:

combined_data = (
    pd.read_parquet("../data/train.parquet")
    .sort_values("date")
    .reset_index(drop=True)
)

combined_data["usd_zar_28_movement"] = (
    combined_data["usd_zar_28"] - combined_data["usd_zar"]
)
combined_data.head(1)

,date,gold_usd_per_oz,platinum_usd_per_oz,gold_usd_per_oz_return,platinum_usd_per_oz_return,us_fed_funds,us_5y_yield,vix,broad_usd_index,iron_ore_usd_per_tonne,...,usd_zar,usd_zar_28,usd_zar_1w_return,usd_zar_1m_return,usd_zar_3m_return,usd_zar_1m_volatility,richards_bay_coal_usd,sa_5y_cds_bp,sa_5y_yield,usd_zar_28_movement
0,2008-10-10,855.400024,996.700012,-0.019289,-0.007625,0.79,2.77,69.95,97.999,60.8,...,9.3626,10.0284,0.085265,0.066904,-0.004945,0.043376,112.4,455.4,9.115,0.6658


In [3]:
X_all = combined_data.drop(columns=["date", "usd_zar", "usd_zar_28_movement"])

In [57]:
y = combined_data["usd_zar_28_movement"]

In [58]:
model = LGBMRegressor()
fit_cv_timeseries_model(model, X_all, y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000111 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2909
[LightGBM] [Info] Number of data points in the train set: 535, number of used features: 22
[LightGBM] [Info] Start training from score -0.118947
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

np.float64(0.6025146828137833)

Engineered data

In [ ]:

engineered_data = combined_data.copy().drop(columns=["date", "usd_zar_28", "usd_zar_28_movement"])
engineered_data["interest_rate_diff"] = (
    engineered_data["sa_repo_rate"] - engineered_data["us_fed_funds"]
)
engineered_data["sa_us_5y_yield_spread"] = (
    engineered_data["sa_5y_yield"] - engineered_data["us_5y_yield"]
)

engineered_data["commodities"] = np.mean([engineered_data.iron_ore_usd_per_tonne, 
engineered_data.gold_usd_per_oz, engineered_data.platinum_usd_per_oz, engineered_data.richards_bay_coal_usd], axis=0)

engineered_features = engineered_data.columns
# [
#     #"commodities",
#     #"usd_zar",
#     #"gold_usd_per_oz",
#     #"platinum_usd_per_oz",
#     #"richards_bay_coal_usd",
#     #"iron_ore_usd_per_tonne",
#     "brent_usd_per_barrel",
#     "interest_rate_diff",
#     "sa_us_5y_yield_spread",
#     "sa_yoy_inflation",
#     "sa_5y_cds_bp",
#     "vix",
#     "broad_usd_index",
#     "sa_cpi",
#     #"gold_usd_per_oz_return",
#     #"platinum_usd_per_oz_return",
#     #"usd_zar_1w_return",
#     #"usd_zar_1m_return",
#     #"usd_zar_3m_return",
#     #"usd_zar_1m_volatility",
# ]

X_engineered = engineered_data[engineered_features]
assert X_engineered.select_dtypes(exclude="number").empty

In [37]:
model = LGBMRegressor()
fit_cv_timeseries_model(model, X_engineered, y)

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000134 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3335
[LightGBM] [Info] Number of data points in the train set: 535, number of used features: 25
[LightGBM] [Info] Start training from score -0.118947
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, 

np.float64(0.7272912203708323)

Backward stepwise feature selection

In [ ]:
from sklearn.base import clone
from sklearn.model_selection import TimeSeriesSplit, cross_val_score


def backward_stepwise_selection(estimator, X, y, min_features=1, tolerance=0.0):
    """Remove one feature at a time when doing so improves time-series CV RMSE."""
    selected_features = list(X.columns)
    cv = TimeSeriesSplit()

    def cv_rmse(features):
        scores = cross_val_score(
            clone(estimator),
            X[features],
            y,
            cv=cv,
            scoring="neg_root_mean_squared_error",
            n_jobs=-1,
        )
        return -scores.mean()

    current_rmse = cv_rmse(selected_features)
    history = [{"removed": None, "n_features": len(selected_features), "cv_rmse": current_rmse}]

    while len(selected_features) > min_features:
        candidates = []
        for feature in selected_features:
            remaining = [name for name in selected_features if name != feature]
            candidates.append((cv_rmse(remaining), feature))

        candidate_rmse, feature_to_remove = min(candidates)
        if candidate_rmse > current_rmse - tolerance:
            break

        selected_features.remove(feature_to_remove)
        current_rmse = candidate_rmse
        history.append(
            {
                "removed": feature_to_remove,
                "n_features": len(selected_features),
                "cv_rmse": current_rmse,
            }
        )
        print(f"Removed {feature_to_remove}; CV RMSE: {current_rmse:.6f}")

    return selected_features, pd.DataFrame(history)


In [ ]:
selection_model = LGBMRegressor(random_state=42, verbosity=-1)
best_features, selection_history = backward_stepwise_selection(
    selection_model,
    X_engineered,
    y,
)

print(f"Selected {len(best_features)} of {X_engineered.shape[1]} features")
print(best_features)
selection_history

Regularized XGBoost with selected features

In [ ]:
from xgboost import XGBRegressor


regularized_xgb_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    learning_rate=0.03,
    max_depth=3,
    min_child_weight=5,
    gamma=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.5,
    reg_lambda=2.0,
    random_state=42,
    n_jobs=-1,
)
fit_cv_timeseries_model(regularized_xgb_model, X_engineered[best_features], y)